In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

START = "2025-04-01"      # risk index reliable from here
WAR_START = "2026-02-28"

# Point-in-time features (geopolitics, inventories, positioning, growth, weather, gas)
f = spark.table("oil_shock.gold.fct_model_features_daily").select(
    "price_date", "brent_ret_1d",
    "gulf_risk_surprise_z_3d_max", "redsea_risk_surprise_z_3d_max", "russia_risk_surprise_z_3d_max",
    "gulf_talks_z_3d_max", "crude_stocks_commercial_kbbl", "brent_mm_net_pct_oi", "wti_mm_net_pct_oi",
    "china_cli", "us_cli", "us_industrial_production_yoy_pct", "storms_in_gulf", "hurricane_in_gulf",
    "henry_hub").toPandas()

# Same-day market drivers (dollar, rates, fear)
m = spark.table("oil_shock.gold.fct_macro_daily").select(
    "obs_date", "usd_index", "treasury_10y_yield", "fed_funds_rate", "vix_model_only").toPandas()

# Same-day chokepoint transits (market sees these via tanker tracking)
c = spark.sql("""
    SELECT transit_date,
           MAX(CASE WHEN lower(chokepoint_name) LIKE '%hormuz%' THEN ships_tanker_pct_of_2025 END) AS hormuz_pct,
           MAX(CASE WHEN lower(chokepoint_name) LIKE '%mandeb%' THEN ships_tanker_pct_of_2025 END) AS mandeb_pct,
           MAX(CASE WHEN lower(chokepoint_name) LIKE '%suez%'   THEN ships_tanker_pct_of_2025 END) AS suez_pct
    FROM oil_shock.gold.fct_chokepoint_daily
    GROUP BY transit_date
""").toPandas()

for df, col in ((f, "price_date"), (m, "obs_date"), (c, "transit_date")):
    df[col] = pd.to_datetime(df[col])

df = f[f["price_date"] >= "2025-03-01"].sort_values("price_date")
df = pd.merge_asof(df, m.sort_values("obs_date"), left_on="price_date", right_on="obs_date", direction="backward")
df = pd.merge_asof(df, c.sort_values("transit_date"), left_on="price_date", right_on="transit_date", direction="backward")

levels = ["crude_stocks_commercial_kbbl", "brent_mm_net_pct_oi", "wti_mm_net_pct_oi", "china_cli", "us_cli",
          "us_industrial_production_yoy_pct", "henry_hub", "usd_index", "treasury_10y_yield",
          "fed_funds_rate", "vix_model_only", "hormuz_pct", "mandeb_pct", "suez_pct"]
df[levels] = df[levels].ffill()

df["d_hormuz"] = df["hormuz_pct"].diff()
df["d_mandeb"] = df["mandeb_pct"].diff()
df["d_suez"] = df["suez_pct"].diff()
df["d_crude_stocks"] = df["crude_stocks_commercial_kbbl"].diff()
df["d_brent_mm"] = df["brent_mm_net_pct_oi"].diff()
df["d_wti_mm"] = df["wti_mm_net_pct_oi"].diff()
df["d_ln_usd"] = np.log(df["usd_index"]).diff()
df["d_10y"] = df["treasury_10y_yield"].diff()
df["d_fed"] = df["fed_funds_rate"].diff()
df["d_vix"] = df["vix_model_only"].diff()
df["d_china_cli"] = df["china_cli"].diff()
df["d_us_cli"] = df["us_cli"].diff()
df["d_us_ip"] = df["us_industrial_production_yoy_pct"].diff()
df["d_ln_hh"] = np.log(df["henry_hub"].where(df["henry_hub"] > 0)).diff()
df["hurricane"] = df["hurricane_in_gulf"].astype(float)

GROUPS = {
    "Geopolitics": ["gulf_risk_surprise_z_3d_max", "redsea_risk_surprise_z_3d_max",
                    "russia_risk_surprise_z_3d_max", "gulf_talks_z_3d_max"],
    "Flows": ["d_hormuz", "d_mandeb", "d_suez"],
    "Inventories": ["d_crude_stocks"],
    "Positioning": ["d_brent_mm", "d_wti_mm"],
    "Dollar and rates": ["d_ln_usd", "d_10y", "d_fed"],
    "Fear": ["d_vix"],
    "Growth": ["d_china_cli", "d_us_cli", "d_us_ip"],
    "Weather and gas": ["storms_in_gulf", "hurricane", "d_ln_hh"],
}
FEATURES = [x for g in GROUPS.values() for x in g]
FEATURE_GROUP = {x: g for g, xs in GROUPS.items() for x in xs}

df = df[(df["price_date"] >= START) & df["brent_ret_1d"].notna()].copy()
df[FEATURES] = df[FEATURES].fillna(0)
df["regime"] = np.where(df["price_date"] >= WAR_START, "war", "pre_war")
ALPHAS = np.logspace(-2, 3, 30)

fit_rows, coef_rows, daily_parts = [], [], []
for reg, d in df.groupby("regime"):
    X, y = d[FEATURES].values, d["brent_ret_1d"].values
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    model = RidgeCV(alphas=ALPHAS).fit(Xs, y)

    preds, actual = [], []
    for tr, te in TimeSeriesSplit(n_splits=5).split(Xs):
        sc = StandardScaler().fit(X[tr])
        mdl = RidgeCV(alphas=ALPHAS).fit(sc.transform(X[tr]), y[tr])
        preds.append(mdl.predict(sc.transform(X[te])))
        actual.append(y[te])

    fit_rows.append({"regime": reg, "days": len(d), "first_day": d["price_date"].min().date(),
                     "last_day": d["price_date"].max().date(), "alpha": round(model.alpha_, 3),
                     "r2_in_sample": round(model.score(Xs, y), 3),
                     "r2_held_out": round(r2_score(np.concatenate(actual), np.concatenate(preds)), 3)})

    for feat, coef in zip(FEATURES, model.coef_):
        coef_rows.append({"regime": reg, "group": FEATURE_GROUP[feat], "feature": feat,
                          "pp_per_1sd_move": round(100 * coef, 3)})

    contrib = Xs * model.coef_
    part = pd.DataFrame({"price_date": d["price_date"].values, "regime": reg,
                         "actual_pct": 100 * y, "average_day_pct": 100 * model.intercept_})
    for g, xs in GROUPS.items():
        idx = [FEATURES.index(x) for x in xs]
        part[g] = 100 * contrib[:, idx].sum(axis=1)
    part["Unexplained"] = 100 * (y - model.predict(Xs))
    daily_parts.append(part)

fit = pd.DataFrame(fit_rows)
coefs = pd.DataFrame(coef_rows)
daily = pd.concat(daily_parts, ignore_index=True).sort_values("price_date")

PARTS = list(GROUPS) + ["Unexplained"]
daily["week_start"] = daily["price_date"] - pd.to_timedelta(daily["price_date"].dt.dayofweek, unit="D")
weekly = daily.groupby(["week_start", "regime"])[["actual_pct", "average_day_pct"] + PARTS].sum().reset_index()

share = (weekly.groupby("regime")[PARTS].apply(lambda x: x.abs().mean())
         .apply(lambda r: 100 * r / r.sum(), axis=1).round(1).T.reset_index()
         .rename(columns={"index": "driver"}))

print("Fit (R² = share of daily Brent moves explained):")
display(fit)
print("Share of weekly movement by driver (%):")
display(share)
print("Last 8 weeks (percentage points of Brent's weekly move):")
display(weekly.tail(8).round(2))
print("Coefficients (pp of daily Brent move per 1 standard-deviation driver move):")
display(coefs.sort_values(["regime", "group"]))

In [0]:
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

NOTES = {
    "Geopolitics": "News-based risk and talks signals (GDELT). Association, not proof of cause.",
    "Flows": "Daily tanker transits vs 2025 normal (IMF PortWatch).",
    "Inventories": "EIA weekly US crude stock change; US-only signal.",
    "Positioning": "Fund net positions (CFTC, ICE). Funds can follow prices as well as lead them.",
    "Dollar and rates": "Moves with oil (two-way): in the war, the oil shock itself lifts yields. Not a cause.",
    "Fear": "VIX changes (model input only; Cboe via FRED).",
    "Growth": "OECD leading indicators (China, US) and US industrial production.",
    "Weather and gas": "Gulf of Mexico storms and Henry Hub gas.",
    "Unexplained": "Not captured by public drivers; includes noise in the physical Brent price.",
    "Average day": "Average daily drift in the period (the war-time uptrend).",
}

def to_long(wide, keys):
    long = wide.rename(columns={"average_day_pct": "Average day"}).melt(
        id_vars=keys + ["actual_pct"], value_vars=["Average day"] + PARTS,
        var_name="driver", value_name="contribution_pp")
    long["driver_note"] = long["driver"].map(NOTES)
    return long

daily_long = to_long(daily.drop(columns=["week_start"]), ["price_date", "regime"])
weekly_long = to_long(weekly, ["week_start", "regime"])
share_long = share.melt(id_vars="driver", var_name="regime", value_name="share_pct")
share_long["driver_note"] = share_long["driver"].map(NOTES)

tables = {"driver_attribution_daily": daily_long,
          "driver_attribution_weekly": weekly_long,
          "driver_attribution_share": share_long,
          "driver_attribution_fit": fit.astype({"first_day": str, "last_day": str}),
          "driver_attribution_coefs": coefs}
for name, out in tables.items():
    out = out.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-driver-attribution")
with mlflow.start_run(run_name=f"driver_attribution {run_stamp}"):
    mlflow.log_params({
        "model": "RidgeCV on standardized daily drivers, separate pre-war and war fits",
        "target": "EIA Brent physical spot daily log return",
        "start": START, "war_start": WAR_START,
        "held_out_method": "TimeSeriesSplit(5)",
        "same_day_inputs": "dollar, rates, VIX, chokepoint transits",
        "point_in_time_inputs": "risk index, EIA stocks, COT, OECD CLI, US IP, storms, Henry Hub",
        "excluded": "product cracks (derived from Brent)",
    })
    for _, r in fit.iterrows():
        mlflow.log_metrics({f"{r['regime']}_r2_in_sample": float(r["r2_in_sample"]),
                            f"{r['regime']}_r2_held_out": float(r["r2_held_out"]),
                            f"{r['regime']}_alpha": float(r["alpha"]),
                            f"{r['regime']}_days": int(r["days"])})
    for name in ["driver_attribution_share", "driver_attribution_fit", "driver_attribution_coefs"]:
        path = f"/tmp/{name}.csv"
        tables[name].to_csv(path, index=False)
        mlflow.log_artifact(path)
print("MLflow run logged.")

In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

START, WAR_START = "2025-04-01", "2026-02-28"
ALPHAS = np.logspace(-2, 3, 30)

# ---------- Load ----------
f = spark.table("oil_shock.gold.fct_model_features_daily").select(
    "price_date", "brent", "wti", "brent_vol_20d", "crude_stocks_commercial_kbbl",
    "brent_mm_net_pct_oi", "wti_mm_net_pct_oi", "china_cli", "us_cli",
    "us_industrial_production_yoy_pct", "storms_in_gulf", "hurricane_in_gulf", "henry_hub"
).where("price_date >= '2025-01-01'").toPandas()
m = spark.table("oil_shock.gold.fct_macro_daily").select(
    "obs_date", "usd_index", "treasury_10y_yield", "fed_funds_rate", "vix_model_only").toPandas()
c = spark.sql("""
    SELECT transit_date,
      MAX(CASE WHEN lower(chokepoint_name) LIKE '%hormuz%' THEN 100 * ships_tanker / nullif(ships_tanker_2025_avg, 0) END) AS hormuz_daily_pct,
      MAX(CASE WHEN lower(chokepoint_name) LIKE '%mandeb%' THEN 100 * ships_tanker / nullif(ships_tanker_2025_avg, 0) END) AS mandeb_daily_pct,
      MAX(CASE WHEN lower(chokepoint_name) LIKE '%suez%'   THEN 100 * ships_tanker / nullif(ships_tanker_2025_avg, 0) END) AS suez_daily_pct
    FROM oil_shock.gold.fct_chokepoint_daily
    GROUP BY transit_date
""").toPandas()
news = spark.sql("""
    SELECT news_date, event_region, category, stories_zscore
    FROM oil_shock.gold.fct_news_category_daily
    WHERE event_region IN ('Gulf', 'Red Sea', 'Russia/Black Sea') AND news_date >= '2025-01-01'
""").toPandas()

for df_, col in ((f, "price_date"), (m, "obs_date"), (c, "transit_date"), (news, "news_date")):
    df_[col] = pd.to_datetime(df_[col])
f = f.sort_values("price_date").reset_index(drop=True)

# ---------- News: strongest surprise since the previous trading day ----------
NEWS_KEYS = ["gulf_vessel_attack_or_seizure", "gulf_infrastructure_strike", "gulf_chokepoint",
             "gulf_sanctions", "gulf_opec_plus", "gulf_talks_or_ceasefire",
             "redsea_vessel_attack_or_seizure", "redsea_infrastructure_strike",
             "russia_vessel_attack_or_seizure", "russia_infrastructure_strike", "russia_sanctions"]
short = {"Gulf": "gulf", "Red Sea": "redsea", "Russia/Black Sea": "russia"}
news["key"] = news["event_region"].map(short) + "_" + news["category"]
news = news[news["key"].isin(NEWS_KEYS)]
tdays = f["price_date"].values
pos = np.searchsorted(tdays, news["news_date"].values)
news = news[pos < len(tdays)].copy()
news["price_date"] = tdays[pos[pos < len(tdays)]]
news_wide = (news.groupby(["price_date", "key"])["stories_zscore"].max()
             .unstack().reindex(columns=NEWS_KEYS).reset_index())
df = f.merge(news_wide, on="price_date", how="left")
df[NEWS_KEYS] = df[NEWS_KEYS].fillna(0)

# ---------- Markets and flows (same day) ----------
df = pd.merge_asof(df, m.sort_values("obs_date"), left_on="price_date", right_on="obs_date", direction="backward")
df = pd.merge_asof(df, c.sort_values("transit_date"), left_on="price_date", right_on="transit_date", direction="backward")
levels = ["crude_stocks_commercial_kbbl", "brent_mm_net_pct_oi", "wti_mm_net_pct_oi", "china_cli", "us_cli",
          "us_industrial_production_yoy_pct", "henry_hub", "usd_index", "treasury_10y_yield", "fed_funds_rate",
          "vix_model_only", "hormuz_daily_pct", "mandeb_daily_pct", "suez_daily_pct"]
df[levels] = df[levels].ffill()

df["d_hormuz"] = df["hormuz_daily_pct"].diff()
df["d_mandeb"] = df["mandeb_daily_pct"].diff()
df["d_suez"] = df["suez_daily_pct"].diff()
df["d_crude_stocks"] = df["crude_stocks_commercial_kbbl"].diff()
df["d_funds"] = (df["brent_mm_net_pct_oi"].diff() + df["wti_mm_net_pct_oi"].diff()) / 2
df["d_china_cli"] = df["china_cli"].diff()
df["d_us_cli"] = df["us_cli"].diff()
df["d_us_ip"] = df["us_industrial_production_yoy_pct"].diff()
df["d_ln_hh"] = np.log(df["henry_hub"].where(df["henry_hub"] > 0)).diff()
df["hurricane"] = df["hurricane_in_gulf"].astype(float)
df["d_ln_usd"] = np.log(df["usd_index"]).diff()
df["d_10y"] = df["treasury_10y_yield"].diff()
df["d_fed"] = df["fed_funds_rate"].diff()
df["d_vix"] = df["vix_model_only"].diff()

# ---------- Targets: daily log returns, extreme days capped at 4x normal volatility ----------
df["brent_ret"] = np.log(df["brent"]).diff()
df["wti_ret"] = np.log(df["wti"].where(df["wti"] > 0)).diff()
df["wti_vol_20d"] = df["wti_ret"].shift(1).rolling(20, min_periods=10).std()
df["brent_clip"] = df["brent_ret"].clip(-4 * df["brent_vol_20d"], 4 * df["brent_vol_20d"])
df["wti_clip"] = df["wti_ret"].clip(-4 * df["wti_vol_20d"], 4 * df["wti_vol_20d"])

GROUPS = {
    "Attacks and strikes": ["gulf_vessel_attack_or_seizure", "gulf_infrastructure_strike",
                            "redsea_vessel_attack_or_seizure", "redsea_infrastructure_strike",
                            "russia_vessel_attack_or_seizure", "russia_infrastructure_strike"],
    "Chokepoint news": ["gulf_chokepoint"],
    "Sanctions and OPEC+": ["gulf_sanctions", "gulf_opec_plus", "russia_sanctions"],
    "Talks and ceasefires": ["gulf_talks_or_ceasefire"],
    "Flows": ["d_hormuz", "d_mandeb", "d_suez"],
    "Inventories": ["d_crude_stocks"],
    "Positioning": ["d_funds"],
    "Growth": ["d_china_cli", "d_us_cli", "d_us_ip"],
    "Weather and gas": ["storms_in_gulf", "hurricane", "d_ln_hh"],
    "Co-moving markets": ["d_ln_usd", "d_10y", "d_fed", "d_vix"],
}
base_feats = [x for xs in GROUPS.values() for x in xs]
for x in base_feats:
    df[f"{x}_lag1"] = df[x].shift(1)
GROUP_FEATS = {g: xs + [f"{x}_lag1" for x in xs] for g, xs in GROUPS.items()}
FULL = [x for xs in GROUP_FEATS.values() for x in xs]
CORE = [x for g, xs in GROUP_FEATS.items() if g != "Co-moving markets" for x in xs]
FEATURE_GROUP = {x: g for g, xs in GROUP_FEATS.items() for x in xs}

df = df[(df["price_date"] >= START) & df["brent_ret"].notna()].copy()
df[FULL] = df[FULL].fillna(0)
df["regime"] = np.where(df["price_date"] >= WAR_START, "war", "pre_war")
print("Days capped (Brent):", int((df["brent_clip"] != df["brent_ret"]).sum()),
      "| (WTI):", int((df["wti_clip"] != df["wti_ret"]).sum()))

# ---------- Fit ----------
def fit_regime(d, feats, target):
    d = d[d[target].notna()]
    X, y = d[feats].values, d[target].values
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    model = RidgeCV(alphas=ALPHAS).fit(Xs, y)
    preds, actual, fold_coefs = [], [], []
    for tr, te in TimeSeriesSplit(n_splits=5).split(Xs):
        s = StandardScaler().fit(X[tr])
        mdl = RidgeCV(alphas=ALPHAS).fit(s.transform(X[tr]), y[tr])
        preds.append(mdl.predict(s.transform(X[te])))
        actual.append(y[te])
        fold_coefs.append(mdl.coef_)
    return {"d": d, "model": model, "Xs": Xs, "y": y, "r2_in": model.score(Xs, y),
            "r2_ho": r2_score(np.concatenate(actual), np.concatenate(preds)),
            "fold_coefs": np.array(fold_coefs)}

CONFIGS = [("v2 core (Brent)", "brent_clip", CORE),
           ("v2 full (Brent)", "brent_clip", FULL),
           ("v2 full (WTI check)", "wti_clip", FULL)]
results, fit_rows = {}, []
for name, target, feats in CONFIGS:
    for reg, d in df.groupby("regime"):
        r = fit_regime(d, feats, target)
        results[(name, reg)] = r
        fit_rows.append({"version": name, "regime": reg, "days": len(r["y"]),
                         "alpha": round(r["model"].alpha_, 2),
                         "r2_in_sample": round(r["r2_in"], 3), "r2_held_out": round(r["r2_ho"], 3)})

v1 = spark.table("oil_shock.ml.driver_attribution_fit").toPandas()
v1_rows = [{"version": "v1 (Brent)", "regime": r["regime"], "days": int(r["days"]), "alpha": r["alpha"],
            "r2_in_sample": r["r2_in_sample"], "r2_held_out": r["r2_held_out"]} for _, r in v1.iterrows()]
fit_v2 = pd.DataFrame(v1_rows + fit_rows).sort_values(["regime", "version"])

# ---------- Contributions (v2 full, Brent) ----------
parts = []
for reg in ["pre_war", "war"]:
    r = results[("v2 full (Brent)", reg)]
    contrib = r["Xs"] * r["model"].coef_
    p = pd.DataFrame({"price_date": r["d"]["price_date"].values, "regime": reg,
                      "actual_pct": 100 * r["d"]["brent_ret"].values,
                      "Average day": 100 * r["model"].intercept_})
    for g, xs in GROUP_FEATS.items():
        p[g] = 100 * contrib[:, [FULL.index(x) for x in xs]].sum(axis=1)
    p["Unexplained"] = p["actual_pct"] - p[["Average day"] + list(GROUP_FEATS)].sum(axis=1)
    parts.append(p)
daily_v2 = pd.concat(parts, ignore_index=True).sort_values("price_date")
PARTS_V2 = list(GROUP_FEATS) + ["Unexplained"]
daily_v2["week_start"] = daily_v2["price_date"] - pd.to_timedelta(daily_v2["price_date"].dt.dayofweek, unit="D")
weekly_v2 = daily_v2.groupby(["week_start", "regime"])[["actual_pct", "Average day"] + PARTS_V2].sum().reset_index()
share_v2 = (weekly_v2.groupby("regime")[PARTS_V2].apply(lambda x: x.abs().mean())
            .apply(lambda r: 100 * r / r.sum(), axis=1).round(1).T.reset_index()
            .rename(columns={"index": "driver"}))

# ---------- Direction stability (war, v2 full Brent) ----------
r = results[("v2 full (Brent)", "war")]
full_coef = r["model"].coef_
stab = pd.DataFrame({
    "feature": FULL, "group": [FEATURE_GROUP[x] for x in FULL],
    "pp_per_1sd_move": np.round(100 * full_coef, 3),
    "same_sign_folds_of_5": [int((np.sign(r["fold_coefs"][:, i]) == np.sign(full_coef[i])).sum())
                             for i in range(len(FULL))]})
stab["abs"] = stab["pp_per_1sd_move"].abs()
stab = stab.sort_values("abs", ascending=False).drop(columns="abs").head(15)

print("Fit: v1 vs v2 (R² = share of daily moves explained)")
display(fit_v2)
print("Share of weekly movement by driver, v2 full (%)")
display(share_v2)
print("War: 15 strongest drivers and direction stability")
display(stab)

In [0]:
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# ---------- 1) Rolling 60-day correlations with Brent ----------
df["gulf_attacks_news"] = df[["gulf_vessel_attack_or_seizure", "gulf_infrastructure_strike"]].max(axis=1)
df["redsea_attacks_news"] = df[["redsea_vessel_attack_or_seizure", "redsea_infrastructure_strike"]].max(axis=1)
CORR_DRIVERS = {
    "gulf_attacks_news": ("Attacks and strikes", "Gulf attack and strike news (surprise)"),
    "redsea_attacks_news": ("Attacks and strikes", "Red Sea attack and strike news (surprise)"),
    "gulf_chokepoint": ("Chokepoint news", "Gulf chokepoint news (surprise)"),
    "gulf_sanctions": ("Sanctions and OPEC+", "Gulf sanctions news (surprise)"),
    "gulf_talks_or_ceasefire": ("Talks and ceasefires", "Gulf talks and ceasefire news (surprise)"),
    "d_hormuz": ("Flows", "Change in Hormuz tanker transits"),
    "d_suez": ("Flows", "Change in Suez tanker transits"),
    "d_crude_stocks": ("Inventories", "EIA weekly US crude stock change"),
    "d_funds": ("Positioning", "Change in fund net positions"),
    "d_ln_usd": ("Co-moving markets", "US dollar index change"),
    "d_10y": ("Co-moving markets", "US 10-year yield change"),
    "d_vix": ("Co-moving markets", "VIX change (model input only)"),
}
WINDOW, MIN_OBS = 60, 40
clear_level = 2 / np.sqrt(WINDOW)   # rough 95% band for a correlation over 60 days

dd = df.sort_values("price_date").set_index("price_date")
corr_rows = []
for col, (grp, label) in CORR_DRIVERS.items():
    rc = dd["brent_clip"].rolling(WINDOW, min_periods=MIN_OBS).corr(dd[col])
    tmp = rc.dropna().reset_index()
    tmp.columns = ["price_date", "correlation"]
    tmp["driver"], tmp["group"], tmp["driver_label"] = col, grp, label
    corr_rows.append(tmp)
corr = pd.concat(corr_rows, ignore_index=True)
corr["correlation"] = corr["correlation"].round(3)
corr["is_clearly_nonzero"] = corr["correlation"].abs() >= clear_level

# ---------- 2) Direction table (v2 full Brent, both periods, all drivers) ----------
dir_rows = []
for reg in ["pre_war", "war"]:
    r = results[("v2 full (Brent)", reg)]
    for i, feat in enumerate(FULL):
        coef = r["model"].coef_[i]
        dir_rows.append({
            "regime": reg, "feature": feat, "group": FEATURE_GROUP[feat],
            "timing": "previous day" if feat.endswith("_lag1") else "same day",
            "direction": "pushes up" if coef > 0 else ("pushes down" if coef < 0 else "none"),
            "pp_per_1sd_move": round(100 * coef, 3),
            "same_sign_folds_of_5": int((np.sign(r["fold_coefs"][:, i]) == np.sign(coef)).sum()),
        })
direction = pd.DataFrame(dir_rows)
direction["is_stable"] = direction["same_sign_folds_of_5"] >= 4

# Latest correlation snapshot for the check below
latest = (corr.sort_values("price_date").groupby("driver").tail(1)
          [["driver_label", "group", "price_date", "correlation", "is_clearly_nonzero"]]
          .sort_values("correlation"))

# ---------- 3) Save ----------
tables = {"driver_correlation_rolling": corr,
          "driver_direction": direction,
          "driver_attribution_fit_versions": fit_v2}
for name, out in tables.items():
    out = out.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

# ---------- 4) Log v2 (including the kill-rule outcome) ----------
war_ho = float(fit_v2.loc[(fit_v2["version"] == "v2 full (Brent)") & (fit_v2["regime"] == "war"), "r2_held_out"].iloc[0])
mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-driver-attribution")
with mlflow.start_run(run_name=f"driver_attribution v2 {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/41_driver_attribution", "project": "oil-shock-2026",
                     "author": "Zia Malik", "version": "v2",
                     "kill_rule": "war held-out R2 >= 0.10 and stable directions",
                     "kill_rule_outcome": "TRIGGERED: percentage split dropped" if war_ho < 0.10 else "passed"})
    mlflow.log_params({"model": "RidgeCV, news by type + previous-day values, core and full layers",
                       "target": "EIA Brent daily log return, capped at 4x 20-day volatility",
                       "check_target": "WTI", "corr_window_days": WINDOW, "features_full": len(FULL)})
    for _, rr in fit_v2.iterrows():
        key = rr["version"].replace(" ", "_").replace("(", "").replace(")", "")
        mlflow.log_metrics({f"{key}_{rr['regime']}_r2_in": float(rr["r2_in_sample"]),
                            f"{key}_{rr['regime']}_r2_held_out": float(rr["r2_held_out"])})
    for name, out in tables.items():
        path = f"/tmp/{name}.csv"
        out.to_csv(path, index=False)
        mlflow.log_artifact(path)
print("MLflow run logged. Kill rule:", "TRIGGERED" if war_ho < 0.10 else "passed")

print(f"Latest 60-day correlations with Brent (clearly nonzero if |r| >= {clear_level:.2f}):")
display(latest)

In [0]:
# Weekly guard: in the scheduled job, run only on Monday's morning run (Dubai time).
# For a manual run on another day, set force_weekly = true.
from datetime import datetime
from zoneinfo import ZoneInfo

dbutils.widgets.dropdown("force_weekly", "false", ["false", "true"])
_now = datetime.now(ZoneInfo("Asia/Dubai"))
if not ((_now.weekday() == 0 and _now.hour < 12) or dbutils.widgets.get("force_weekly") == "true"):
    dbutils.notebook.exit(f"Skipped: weekly model, runs Monday mornings (now {_now:%a %H:%M} Dubai)")